In [31]:
# this requires:
# 1. setting up account on CDS cds.copernicus-climate.eu
# installation of python cdsapi library
# creation of .ecmwfapi file with credentials from the CDS account

import cdsapi
import xarray as xr
import os, glob, sys
import pandas as pd
import numpy as np
import datetime

In [66]:
def download_forecast(outputfile,year, month):

    #simiple function to download monthly rainfall forecast by ECMWF SEAS5.1 model
    
    # forecast system, variable, definition of region in hardcoded
    
    #this defines dataset
    dataset = "seasonal-monthly-single-levels"

    #this defines dataset dictionary
    request = {
        "originating_centre": "ecmwf",
        "system": "51",
        "variable": ["total_precipitation"],
        "year": [str(year)],
        "month": [str(month).zfill(2)],
        "leadtime_month": [
            "1",
            "2",
            "3",
            "4",
            "5",
            "6"
        ],
        "data_format": "netcdf",
        "product_type": ["monthly_mean"],
        "area": [40, 60, 25, 70]
    }


    client = cdsapi.Client()

    try:
        #requesting data
        client.retrieve(dataset, request, outputfile)
    except Exception as e:
        print(e)
        print("Something went wrong. Cannot download data from CDS".format(outputfile))

    #checking if file can be opened
    try:
        ds=xr.open_dataset(outputfile)
        print(ds.dims)
        ds.close()
    except Exception as e:
        print(e)
        print("Something went wrong. Downloaded file cannot be read \n{}".format(outputfile))

    print(f"Successfully downloaded {outputfile}")


In [67]:
# to issue forecast, historical data have to be available for each month till the current month. 
# All files present in the output directory will be read and concatenated 
# and the most recent data will be identified. Download function could then be called to 
# update the stash of data till current date

#data directory
datadir="./data/SEAS51_rainfall/"

files=glob.glob("{}/pr_mon_ECMWF_SEAS51_*.nc".format(datadir))

if len(files)==0:
    print("there appears to be no data files in the {} directory. That should not be the case. Please check if data directory is correct.".format(datadir))
    sys.exit()


dates=[pd.to_datetime(x.split("_")[-1][0:6], format="%Y%m") for x in np.sort(files)]
dates=pd.to_datetime(dates)



#the most recent date available locally
firstdatadate=dates[0]

print(f"last data available on the system: {lastdatadate.strftime('%b %Y')}")

today=pd.to_datetime(datetime.datetime.now()).normalize()
currentday=today.day
currentdate=today.replace(day=1)

if currentday>7:
    lastforecastdate=currentdate
else:
    lastforecastdate=currentdate-pd.offsets.MonthBegin(1)
print(f"last forecast date available on CDS: {lastforecastdate}")

for date in pd.date_range(firstdatadate, lastforecastdate, freq="MS"):
    
    year=date.year
    month=date.month

    #output file
    outputfile="{}/pr_mon_ECMWF_SEAS51_{}{}.nc".format(datadir,year,str(month).zfill(2))
    if not os.path.exists(outputfile):
        print(f"Downloading {outputfile}")
        
        download_forecast(outputfile,year,month)
        
    elif date==lastforecastdate:
        print("Data for the most recent forecast are already available. Exiting...")


last data available on the system: Apr 2026
last forecast date available on CDS: 2026-04-01 00:00:00
Data for the most recent forecast are already available. Exiting...
